# Examen de Sistemas Inteligentes 2024
## Parte práctica
### Nombre del alumno

Pon aquí tu nombre.

***

Instrucciones. Carga el fichero mushroom.csv, y contexta las preguntas debajo de los encabezados.


### Ejercicio 1

Probablemente sea un CNN o un autoencoder. Ambos NN son muy buenos extratores de característica de una imagen. Correctamente configurados poderían reconocer las características indicadas. Tendría 8 salidas y una entrada de 49152 pixeles.

### Ejercicio 2

Explicación del ejercicio 2: La limpieza consiste en eliminar las filas con nulos. Podemos imputarlas (usando media o KNN) pero al ser pocas y tener bastantes datos, podemos eliminarlas sin problemas. Tambien hay que normalizarlas o bien con la normalización estandar (Media / desviación) o con MaxMinScaler ambas son correctas. Otro dato que a la mayoría se os ha olvidado es que las categóricas hay que pasarlas a one-hot-encoding. No hacerlo reduce la nota de este apartado en 0.25.

In [ ]:
# Imports
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.impute import SimpleImputer

# Ver cuantas las filas tienen un valor nulo (NaN) del dataset
mushroomOriginal = pd.read_csv('mushroom.csv')
countNaN = mushroomOriginal.isna().sum()
print(countNaN)

mushroomOriginal = mushroomOriginal.dropna(how='any') # Se eliminan todas las filas que contienen al menos un valor nulo
countNaN = mushroomOriginal.isna().sum()
print(countNaN) # Demostrar que no quedan


# Creo One-Hot (crea columnas por cada categoria de cada columna original)
encoder = OneHotEncoder(sparse_output=False)
display(mushroomOriginal)

# Pasar los valores de la columna  x de 'lo que sea' a numericos
cap_shape=mushroomOriginal["cap-shape"].to_numpy()
gill_attachment=mushroomOriginal["gill-attachment"].to_numpy()
gill_color=mushroomOriginal["gill-color"].to_numpy()
stem_color=mushroomOriginal["stem-color"].to_numpy()

# Crea las nuevas columnas (categoria) dejando valores entre -1 y 1.  
cap_shapeOHE = encoder.fit_transform(cap_shape.reshape(-1, 1))
gill_attachmentOHE = encoder.fit_transform(gill_attachment.reshape(-1, 1))
gill_colorOHE = encoder.fit_transform(gill_color.reshape(-1, 1))
stem_colorOHE = encoder.fit_transform(stem_color.reshape(-1, 1))

# Elimino las columnas originales
mushroomOriginal=mushroomOriginal.drop("cap-shape",axis=1)
mushroomOriginal=mushroomOriginal.drop("gill-attachment",axis=1)
mushroomOriginal=mushroomOriginal.drop("gill-color",axis=1)
mushroomOriginal=mushroomOriginal.drop("stem-color",axis=1)

# Separo la sol que busco
Y = mushroomOriginal["class"].to_numpy() # Paso a numerico (Venenoso no venenoso -> 0 o 1) y lo meto en Y
mushroomOriginal=mushroomOriginal.drop("class",axis=1)  # Elimino del mushroomOriginal la columna que me da la solucion

# Empiezo con aquellas columnas que ya eran numeros (estan como string)
X = mushroomOriginal.to_numpy()
print(X.shape)
# Meto las columnas que he creado con One-Hot (antes categoricas, ahora varias nmericas 0, 1)
X = np.hstack((X,cap_shapeOHE))
X = np.hstack((X,gill_attachmentOHE))
X = np.hstack((X,gill_colorOHE))
X = np.hstack((X,stem_colorOHE))
print(X.shape)

# Se normalizan los valores de X entre 0 y 1 
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Dividir datos (dividir 80% train, 20% test)
print(X)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=13)

cap-diameter       2
cap-shape          0
gill-attachment    0
gill-color         0
stem-height        0
stem-width         2
stem-color         0
season             0
class              0
dtype: int64
cap-diameter       0
cap-shape          0
gill-attachment    0
gill-color         0
stem-height        0
stem-width         0
stem-color         0
season             0
class              0
dtype: int64


,cap-diameter,cap-shape,gill-attachment,gill-color,stem-height,stem-width,stem-color,season,class
0,1371.0,2,2,10,3.807467,1545.0,11,1.804273,1
1,1461.0,2,2,10,3.807467,1557.0,11,1.804273,1
2,1371.0,2,2,10,3.612496,1566.0,11,1.804273,1
3,1261.0,6,2,10,3.787572,1566.0,11,1.804273,1
4,1305.0,6,2,10,3.711971,1464.0,11,0.943195,1
...,...,...,...,...,...,...,...,...,...
54030,73.0,5,3,2,0.887740,569.0,12,0.943195,1
54031,82.0,2,3,2,1.186164,490.0,12,0.943195,1
54032,82.0,5,3,2,0.915593,584.0,12,0.888450,1
54033,79.0,2,3,2,1.034963,491.0,12,0.888450,1


(54031, 4)
(54031, 43)
[[0.72501322 0.99273698 0.43289437 ... 0.         1.         0.        ]
 [0.77260709 0.99273698 0.43625665 ... 0.         1.         0.        ]
 [0.72501322 0.94189582 0.43877837 ... 0.         1.         0.        ]
 ...
 [0.0433633  0.23864218 0.16363127 ... 0.         0.         1.        ]
 [0.04177684 0.26976942 0.13757355 ... 0.         0.         1.        ]
 [0.03807509 0.30193424 0.13785374 ... 0.         0.         1.        ]]


### Ejercicio 3

Aqui el resultado puede variar, pero por lo general, sin tocar algun hiperparametro no se consigue el 97% o superior que he conseguido yo. 
Os pido solo el 90% para dejaros margen.

In [ ]:
# Perceptor Multicapa
from sklearn.neural_network import MLPClassifier # modelo de red neuronal de perceptrón multicapa
from sklearn.metrics import accuracy_score # función para calcular la proporción de predicciones correctas
model = MLPClassifier(hidden_layer_sizes=(20), max_iter=7000, alpha=0.01, learning_rate_init=0.001, random_state=13) 
# 20 : capas ocultas (valor normal) (1 nivel)
# 7000 : numero máximo de iteraciones (épocas) durante el entrenamiento (alto para que converja)
# 0.01 parámetro de regularización L2 (penalizacion moderada)
# tasa de aprendizaje inicial (Tasa más alta (0.01) (Ajuste de Pesos) = aprende más rápido pero puede ser inestable) (Tasa más baja (0.0001) = más estable pero más lento)
# 13 : semilla
model.fit(X_train, y_train)

# Obtener las predicciones del clasificador de scikit-learn
y_pred_sklearn = model.predict(X_test)
ac = accuracy_score(y_pred_sklearn,y_test)
print("Accuracy",ac)

Accuracy 0.9719626168224299


### Ejercicio 4

Idem con esta parte, para conseguir un 98 o 99 se necesitan tocan hiperparametros en la mayoria de las ocasasiones, os pido menos (95%) para tener margen.

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(max_depth=20, random_state=13,n_estimators=400)
# 20 : Límite de profundidad de cada árbol
# 400 : es un numero de arboles robusto
clf.fit(X_train, y_train)

y_predict_RF = clf.predict(X_test)
ac = accuracy_score(y_predict_RF,y_test)
print("Accuracy",ac)

Accuracy 0.9905616729897289


### Ejercicio 5

Aqui la explicación depende de vuestro resultados, pero princpalmente hay que mirar estas cosas:
Los falsos positivos ya que aqui son importantes, si crees que no es venenoso y resulta que lo es puedes morir. Además RF es mas rápido de entrenar y es un pocquito más explicable que una red de neuronas. Dependiendo del resultado se puede argumentar que uno es mejor que otro pero en igualdad o similitud de condiciones, parte con ventaja RF.

In [ ]:
from sklearn.metrics import confusion_matrix

cm_MLP = confusion_matrix(y_test, y_pred_sklearn)
cm_RF = confusion_matrix(y_test, y_predict_RF)
print("Matriz de confusión MLP:")
print(cm_MLP)
print("Matriz de confusión Random Forest:")
print(cm_RF)
# (0,0) TN: Verdaderos negativos
# (0,1) FP: Falsos positivos (¡esto es lo que quieres!)
# (1,0) FN: Falsos negativos
# (1,1) TP: Verdaderos positivos

fp_mlp = cm_MLP[0, 1]
print("Falsos positivos MLP:", fp_mlp)

fp_rf = cm_RF[0, 1]
print("Falsos positivos RF:", fp_rf)